## The Halting Problem & Machine Learning Approximation

### Understanding the Halting Problem
The Halting Problem is a foundational dilemma in computer science, formulated and proved undecidable by Alan Turing in 1936. It asks a simple question: Given an arbitrary computer program and an input, can we create a general algorithm that decides whether the program will finish running (halt) or run forever (infinite loop)?

Turing proved that a perfect, universal program to solve this is **mathematically impossible**. If we try to create a deterministic function $H(P, I)$, it leads to a logical paradox (diagonalization argument) where the analyzer cannot correctly predict its own behavior.

### Mathematical Framework
Let $P$ represent a program and $I$ represent its input. The idealized halting function is defined as:
$$H(P, I) = \begin{cases} 1 & \text{if } P(I) \text{ terminates within finite steps} \\ 0 & \text{if } P(I) \text{ loops infinitely} \end{cases}$$

Since $H(P, I)$ cannot be computed deterministically for all possible programs, we use Machine Learning to build a **probabilistic static analyzer**. We look at structural and complexity metrics of code to approximate the likelihood of a program halting.

We define this as a **Binary Classification** task. For a given feature vector $x \in \mathbb{R}^d$ representing code metrics, the model estimates:
$$\hat{y} = P(Y = 1 \mid x)$$

Where:
* $Y = 1$: The program halts.
* $Y = 0$: The program encounters an infinite loop.

The model is optimized by minimizing the **Binary Cross-Entropy Loss**:
$$\mathcal{L}(y, \hat{y}) = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]$$


In [2]:
# Import core data science and visualization libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import machine learning components
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# Configure runtime environment settings
import warnings
warnings.filterwarnings('ignore')

# Set aesthetic styling for data visualization
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment successfully configured. Ready for data injection.")


Environment successfully configured. Ready for data injection.


In [3]:
# Synthetic Dataset Generation representing code structures
np.random.seed(42)
num_samples = 1500

# Generating static code features:
# - loop_count: Number of iteration loops in the source code
# - nested_depth: Maximum nesting level of loops and conditions
# - cyclomatic_complexity: Number of independent execution paths
# - has_break_condition: Binary indicator (1 = explicit exit condition present, 0 = absent)
# - recursion_depth: Maximum depth of recursive functions
# - lines_of_code: Total physical size of the program

loop_count = np.random.randint(0, 5, num_samples)
nested_depth = np.random.randint(0, 4, num_samples)
cyclomatic_complexity = loop_count * 2 + nested_depth + np.random.randint(1, 10, num_samples)
has_break_condition = np.random.choice([0, 1], size=num_samples, p=[0.3, 0.7])
recursion_depth = np.random.randint(0, 20, num_samples) * np.random.choice([0, 1], size=num_samples, p=[0.8, 0.2])
lines_of_code = cyclomatic_complexity * 5 + np.random.randint(5, 50, num_samples)

# Assemble into a clean DataFrame
df = pd.DataFrame({
    'loop_count': loop_count,
    'nested_depth': nested_depth,
    'cyclomatic_complexity': cyclomatic_complexity,
    'has_break_condition': has_break_condition,
    'recursion_depth': recursion_depth,
    'lines_of_code': lines_of_code
})

# Formulate underlying non-linear logic determining the execution state (with noise)
# High loops, deep nesting, and no break conditions increase risk of infinite execution
score = (df['loop_count'] * 1.8 + df['nested_depth'] * 2.2 + (df['recursion_depth'] > 12) * 2.5) - (df['has_break_condition'] * 4.5)
probability = 1 / (1 + np.exp(score)) # Logistic sigmoid distribution

# Assign classification labels: 1 for Halting, 0 for Infinite Loop
df['halts'] = (probability > 0.5).astype(int)

print(f"Data pipeline complete. Dataset shape: {df.shape}")
print("\nTarget class distribution:")
print(df['halts'].value_counts(normalize=True))


Data pipeline complete. Dataset shape: (1500, 7)

Target class distribution:
halts
0    0.802
1    0.198
Name: proportion, dtype: float64
